# Qwen paired Cedar: CUDA launch correction
Version1 passed58 software checks but stopped during server startup: linker could not find -lcuda. No participant request or Cedar world was started.
This revision adds a private compiler link to the actual installed driver and verifies compilation and driver API loading before the unchanged131-file runtime. It does not load CUDA stubs, change weights, change budgets, or resume the failed launch.
# Qwen alone versus Qwen + native EDI — runtime qualification pending

This is an unexecuted source draft, not a benchmark result. It has received AST/source inspection only.
The19 new boundary tests and39 older tests must execute on Kaggle before model smoke or a fresh Cedar pair.
No new Qwen inference or Cedar run has happened as of preparation.

Owner scope: Kaggle only, fixed Qwen3-VL-8B-Instruct weights, real EDI integration, no teacher/training,
no paid provider, no old-run restart, no previous Cedar solution in model requests.

Required actual hardware: two T4 GPUs and Internet for pinned public weights/packages. Availability/quota is unverified.
The run installs environments, verifies weight/tokenizer hashes, executes boundary tests and two neutral real model
smoke calls, then attempts one fresh paired world. Any failure stops without automatic retries or changed caps.
The local serving process is terminated in finally; ending the Kaggle compute session still requires the platform
job/session lifecycle. A batch Save Version run avoids leaving a successful draft session idle.
Do not Run All on an existing/started attempt.

A real result exists only when actual-pair/comparison.json and native terminal evidence are produced.
A blocked file, passing mock checks or an empty EDI state is not a successful comparison.


In [ ]:
# Source preparation only. Do not repeat a started notebook.
from pathlib import Path
from hashlib import sha256
import base64,io,json,sys,zipfile
source=Path("/kaggle/working/qwen-paired-source-20261028")
if source.exists():raise RuntimeError("Source directory already exists; preserve this attempt, do not Run All again")
inputs=list(Path('/kaggle/input').rglob('edi-qwen-runtime-20261028.bin'))
if len(inputs)!=1:raise RuntimeError('Expected one attached private runtime input')
payload=inputs[0].read_bytes()
assert sha256(payload).hexdigest()=="fd9dddda55f6963b1578d757f3e8765a49ab55acb1a15b094c72cca17b695dfe"
source.mkdir(parents=True,exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
 for name in z.namelist():
  p=Path(name)
  if p.is_absolute() or ".." in p.parts:raise ValueError("Unsafe source member")
  dest=source/p;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(name))
manifest=json.loads((source/"source-manifest.json").read_bytes())
for name,expected in manifest["files"].items():
 assert sha256((source/name).read_bytes()).hexdigest()==expected,name
sys.path[:0]=[str(source),str(source/"src")]
print("QWEN_SOURCE_HASHES_VERIFIED",len(manifest["files"]))
print("New runtime tests and benchmark remain unexecuted until the next cell succeeds.")


In [ ]:
# Kaggle-only CUDA compiler-link qualification. Never loads a stub driver.
from pathlib import Path
import ctypes,hashlib,json,os,subprocess,uuid
if not Path('/kaggle/working').is_dir():raise RuntimeError('Kaggle only')
# Kaggle can inject a working driver through LD_LIBRARY_PATH without ldconfig entries.
actual_driver=ctypes.CDLL('libcuda.so.1')
actual_driver.cuDriverGetVersion.argtypes=[ctypes.POINTER(ctypes.c_int)]
actual_version=ctypes.c_int()
if actual_driver.cuDriverGetVersion(ctypes.byref(actual_version))!=0 or actual_version.value<=0:
 raise RuntimeError('Loaded CUDA driver API unavailable')
found=[]
for mapping in Path('/proc/self/maps').read_text().splitlines():
 fields=mapping.split(maxsplit=5)
 if len(fields)==6 and '/libcuda.so' in fields[5]:
  candidate=Path(fields[5]).resolve(strict=True)
  if '/stubs/' in str(candidate):raise RuntimeError('Stub driver cannot qualify runtime')
  if candidate not in found:found.append(candidate)
if len(found)!=1:raise RuntimeError('Actual loaded CUDA driver mapping is missing or ambiguous')
driver=found[0]
link_dir=Path('/kaggle/temp')/('edi-cuda-link-'+uuid.uuid4().hex[:12])
link_dir.mkdir(parents=True,exist_ok=False)
(link_dir/'libcuda.so').symlink_to(driver)
old_library_path=os.environ.get('LIBRARY_PATH','')
os.environ['LIBRARY_PATH']=str(link_dir)+(os.pathsep+old_library_path if old_library_path else '')
# Do not add link directories or stubs to LD_LIBRARY_PATH.
probe_source='extern int cuDriverGetVersion(int*); int edi_cuda_version(void) { int v=0; return cuDriverGetVersion(&v)==0?v:-1; }'
probe=link_dir/'edi_cuda_link_probe.so'
command=['cc','-shared','-fPIC','-x','c','-','-lcuda','-o',str(probe)]
checked=subprocess.run(command,input=probe_source,text=True,capture_output=True,timeout=60)
receipt={'driver':str(driver),'link_directory':str(link_dir),'compile_returncode':checked.returncode,
 'compiler_stdout':checked.stdout,'compiler_stderr':checked.stderr,'ld_library_path_unchanged':True,
 'runtime_source_unchanged':True,'previous_run':'qwen-cedar-a675b6e46494'}
receipt_path=Path('/kaggle/working/cuda-link-qualification.json')
receipt_path.write_text(json.dumps(receipt,indent=2))
if checked.returncode:raise RuntimeError('CUDA linker qualification failed; see retained receipt')
loaded=ctypes.CDLL(str(probe))
loaded.edi_cuda_version.restype=ctypes.c_int
version=loaded.edi_cuda_version()
receipt['cuda_driver_api_version']=version
receipt['driver_sha256']=hashlib.sha256(driver.read_bytes()).hexdigest()
receipt['passed']=version>0
receipt_path.write_text(json.dumps(receipt,indent=2))
if version<=0:raise RuntimeError('Actual CUDA driver API qualification failed')
print('CUDA_LINK_PREFLIGHT_PASS',json.dumps({'driver':str(driver),'version':version}),flush=True)


In [ ]:
# Owner-authorized Kaggle-only run. Fails closed; no retries/model swaps.
from qwen_kaggle_entry import run
run_directory=run(source)
print("Inspect",run_directory,"for actual-pair/comparison.json and participant reports.")


In [ ]:
# Read actual files only; a missing comparison is not a success.
from pathlib import Path
import json
runs=sorted(Path("/kaggle/working").glob("qwen-cedar-*"))
for run_dir in runs:
 result=run_dir/"actual-pair/comparison.json"
 if result.exists():
  value=json.loads(result.read_bytes())
  print("ACTUAL_TERMINAL_COMPARISON",json.dumps(value,indent=2))
 else:
  print("NO_COMPLETED_COMPARISON",str(run_dir))
